# Premium Forecast — Model Training & UC Registration

Trains an XGBoost regressor to predict:
- `discount_pct` — recommended discount percentage to apply

The adjusted premium is derived at inference time by the serving app using the
live `premium_amount` and `discount_pct` from the `policies` table.

## Inputs
`serverless_stable_mv5gxq_catalog.insurance_trip_wise.ml_premium_features`
(run `create_feature_table` notebook first)

## Output
Registered model: `serverless_stable_mv5gxq_catalog.insurance_trip_wise.premium_forecast_model`

In [0]:
%pip install xgboost scikit-learn mlflow --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

mlflow.set_registry_uri("databricks-uc")

In [0]:
CATALOG  = "serverless_stable_mv5gxq_catalog"
SCHEMA   = "insurance_trip_wise"
VOLUME   = "vl_premium_forecast_tripwise"

EXP_NAME      = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
ARTIFACT_PATH = f"dbfs:/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/mlflow-artifacts"

if mlflow.get_experiment_by_name(EXP_NAME) is None:
    mlflow.create_experiment(name=EXP_NAME, artifact_location=ARTIFACT_PATH)
mlflow.set_experiment(EXP_NAME)

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/<experiment-id>', creation_time=1780430380250, effective_trace_archival_retention=None, experiment_id='<experiment-id>', last_update_time=1780430398893, lifecycle_stage='active', name='/Users/amee.vora@databricks.com/tripwise/notebooks/train_premium_model', tags={'mlflow.experiment.sourceId': '<experiment-id>',
 'mlflow.experiment.sourceName': '/Users/amee.vora@databricks.com/tripwise/notebooks/train_premium_model',
 'mlflow.experiment.sourceType': 'NOTEBOOK_EXPERIMENT',
 'mlflow.experimentType': 'NOTEBOOK',
 'mlflow.ownerEmail': 'amee.vora@databricks.com',
 'mlflow.ownerId': '5683140511648579'}, trace_location=None, workspace='default'>

## 1. Load feature table

In [0]:
features_df = spark.table("serverless_stable_mv5gxq_catalog.insurance_trip_wise.ml_premium_features").toPandas()
print(f"Loaded {len(features_df):,} rows")
features_df.head(3)

Loaded 1,500 rows


,device_id,current_driving_score,total_trips,total_miles,risk_tier,state,has_embedded_policy,trip_count,avg_trip_score,avg_trip_distance_miles,avg_speed_mph,avg_rapid_accel_count,avg_sharp_turn_count,avg_leaving_home_zone_count,avg_rough_road_conditions_count,avg_night_driving_pct,avg_high_risk_area_pct,avg_inclement_weather_pct,total_embedded_conversions,embedded_conversion_rate,policy_id,coverage_level,policy_type,discount_pct
0,DEV0001,78.400002,3,50.299999,medium,PA,True,3,76.70,11.07,22.93,0.333,0.333,0.667,1.333,17.9000,14.9333,24.40,1,0.3333,POL0001_2026,standard,full_term,5.8
1,DEV0002,71.400002,2,42.000000,medium,WA,True,2,70.80,21.25,75.70,1.000,1.000,1.500,1.000,18.8500,16.1500,23.25,1,0.5000,POL0002_2026,standard,full_term,6.4
2,DEV0003,22.400000,6,182.199997,high,TX,True,6,30.37,28.30,44.25,2.500,1.667,3.333,3.833,15.0333,17.1500,21.00,4,0.6667,POL0003_2026,standard,full_term,2.1


## 2. Define features and targets

In [0]:
NUMERIC_FEATURES = [
    "current_driving_score",
    "total_trips",
    "total_miles",
    "avg_trip_score",
    "avg_trip_distance_miles",
    "avg_speed_mph",
    "avg_rapid_accel_count",
    "avg_sharp_turn_count",
    "avg_leaving_home_zone_count",
    "avg_rough_road_conditions_count",
    "avg_night_driving_pct",
    "avg_high_risk_area_pct",
    "avg_inclement_weather_pct",
    "embedded_conversion_rate",
]

CATEGORICAL_FEATURES = [
    "risk_tier",         # low / medium / high
    "state",             # NY, CA, TX, ...
    "coverage_level",    # basic / standard / premium
    "policy_type",       # annual / semi_annual / monthly
]

TARGET = "discount_pct"

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# Drop rows with any nulls in features or target
df = features_df[ALL_FEATURES + [TARGET]].dropna()  # TARGET = discount_pct
print(f"Clean rows for training: {len(df):,}")

X = df[ALL_FEATURES]
y = df[TARGET]

Clean rows for training: 1,500


## 3. Train / test split

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

Train: 1,200  |  Test: 300


## 4. Build preprocessing + model pipeline

In [0]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        (
            "cat",
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
            CATEGORICAL_FEATURES,
        ),
    ]
)

xgb_params = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "n_jobs": -1,
    "tree_method": "hist",
}

model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", XGBRegressor(**xgb_params)),
    ]
)

## 5. Train with MLflow tracking

In [0]:
with mlflow.start_run(run_name="xgboost_discount_v1") as run:

    # --- Train ---
    model_pipeline.fit(X_train, y_train)

    # --- Evaluate discount_pct ---
    discount_pred = model_pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, discount_pred)
    r2  = r2_score(y_test, discount_pred)
    print(f"discount_pct  MAE={mae:.4f}  R²={r2:.4f}")

    # --- Log params & metrics ---
    mlflow.log_params({**xgb_params, "test_size": 0.2, "train_rows": len(X_train)})
    mlflow.log_metrics({
        "mae_discount_pct": round(mae, 4),
        "r2_discount_pct":  round(r2, 4),
    })

    # --- Log feature list as artifact ---
    import json, tempfile, os
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
        json.dump({"numeric": NUMERIC_FEATURES, "categorical": CATEGORICAL_FEATURES}, f, indent=2)
        feature_manifest_path = f.name
    mlflow.log_artifact(feature_manifest_path, artifact_path="metadata")
    os.unlink(feature_manifest_path)

    # --- Define input example for serving ---
    input_example = X_test.iloc[:3]

    # --- Log & register model in Unity Catalog ---
    mlflow.sklearn.log_model(
        sk_model=model_pipeline,
        artifact_path="model",
        input_example=input_example,
        registered_model_name="serverless_stable_mv5gxq_catalog.insurance_trip_wise.premium_forecast_model",
    )

    run_id = run.info.run_id
    print(f"\nRun ID: {run_id}")
    print(f"Registered: serverless_stable_mv5gxq_catalog.insurance_trip_wise.premium_forecast_model")

discount_pct  MAE=1.5337  R²=-0.1210


2026/10/22 20:53:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://<workspace-url>/ml/experiments/<experiment-id>/models/m-c1383a1c973742aa9a692adff06916ea?o=<workspace-id>
/local_disk0/.ephemeral_nfs/envs/pythonEnv-b214441c-7c31-4d1d-b11f-c34bfbab5f6c/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-i

Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '2' of model 'serverless_stable_mv5gxq_catalog.insurance_trip_wise.premium_forecast_model': https://<workspace-url>/explore/data/models/serverless_stable_mv5gxq_catalog/insurance_trip_wise/premium_forecast_model/version/2?o=<workspace-id>



Run ID: <run-id>
Registered: serverless_stable_mv5gxq_catalog.insurance_trip_wise.premium_forecast_model


## 6. Promote the new version to 'Champion' alias

UC model aliases let the serving endpoint always load `@champion`
without changing the endpoint config when you retrain.

In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient(registry_uri="databricks-uc")

# Get the latest version just registered
model_name = "serverless_stable_mv5gxq_catalog.insurance_trip_wise.premium_forecast_model"
versions = client.search_model_versions(f"name='{model_name}'")
latest = sorted(versions, key=lambda v: int(v.version))[-1]

client.set_registered_model_alias(
    name=model_name,
    alias="champion",
    version=latest.version,
)

print(f"Model version {latest.version} promoted to alias 'champion'")

Model version 2 promoted to alias 'champion'


## 7. Quick smoke-test: load champion and predict on one driver

In [0]:
loaded_model = mlflow.sklearn.load_model(f"models:/{model_name}@champion")

sample             = X_test.iloc[[0]]
predicted_discount = loaded_model.predict(sample)[0]

print(f"\nSample driver features:")
print(sample[["current_driving_score", "risk_tier", "coverage_level", "total_miles"]].to_string(index=False))
print(f"\nPredicted discount : {predicted_discount:.1f}%")
print("(Adjusted premium is derived at serving time from the live policy premium)")


Sample driver features:
 current_driving_score risk_tier coverage_level  total_miles
                  29.1      high       standard   162.199997

Predicted discount : 5.4%
(Adjusted premium is derived at serving time from the live policy premium)
